# Wordle strategy explorer

Explore an **optimal tree** — the committed per-opener strategy the solver emits — as a
table of *positions*. Pick an `OPENER` and the cells below enumerate every situation that
strategy can reach: the guesses played so far, the feedback each one got, how many answers
are still possible, and the optimal next guess.

Modelled on `hangman/math/notebooks/strategy_explorer.ipynb` ("All situations at depth k",
"Inspect a tree node"), over Wordle's tree files instead of hangman's.

**Where the data comes from.** `paths.TREES` resolves to the tree directory this machine
should read (the live `local/<solver>/trees/` on a sweep machine, else the committed
`shared/<solver>/trees/`). One gzipped file per opener, one line per answer:

```
nymph 3 salet bbbbb courd bbbbb nymph
```

— the answer, its guess count, then `guess pattern guess pattern ... guess`. There are N
guesses and N-1 patterns, because the last guess *is* the answer and its feedback is
all-green by construction. A pattern is a 5-character string: **`b`** = grey, **`y`** =
yellow, **`g`** = green.

Setting `WORDLE_SOLVER=hardturns` (or `ansturns`) before starting the kernel points every
cell at that sweep instead.

> **The objective these trees optimise is CONSTRAINED.** Each tree minimises the TOTAL
> guesses over the 2315 answers *subject to resolving every answer within `--depth`*. That
> is not the same as minimum expected guesses, and anything read off these positions
> inherits that constraint.

In [1]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'wordle.py').exists():
            return c
    raise FileNotFoundError('Could not find wordle.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
for p in (math_dir, math_dir / 'tools'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import gzip
import numpy as np
import pandas as pd
from IPython.display import display
import wordle                     # the feedback primitives (math/wordle.py)
import paths                      # the canonical solver locations (math/tools/paths.py)

pd.set_option('display.max_rows', 200)   # show up to 200 rows before pandas truncates to a head/tail preview

print(f'imported wordle + paths from {math_dir}')
print(f'solver = {paths.SOLVER!r}')
print(f'trees  = {paths.TREES}')

imported wordle + paths from /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math
solver = 'optturns'
trees  = /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math/data/shared/optturns/trees


## Pick an opener

`OPENER` is any word with a committed tree. The totals come from the sweep's
`opener_totals.csv`: `total` is the summed guesses over all 2315 answers.

In [2]:
OPENER = 'salet'    # <-- the opener whose optimal tree to explore

_totals = paths.load_totals()
_tree_path = Path(paths.TREES) / f'{OPENER}.txt.gz'

print(f'{OPENER!r}: tree {"found" if _tree_path.exists() else "MISSING"} at {_tree_path}')
if OPENER in _totals:
    print(f'  total {_totals[OPENER]:,} guesses over {paths.N_ANSWERS:,} answers'
          f'  ({_totals[OPENER] / paths.N_ANSWERS:.5f} per answer)')
else:
    print(f'  no row in {paths.CSV_PATH} -- not solved (or INFEASIBLE) in this sweep')

'salet': tree found at /Users/patrickliscio/Documents/Ballpark Figures/ballpark-figures/wordle/math/data/shared/optturns/trees/salet.txt.gz
  total 7,920 guesses over 2,315 answers  (3.42117 per answer)


## Every position in the tree

`positions(opener)` returns one row per **position**: a distinct history of
(guess, feedback) pairs the strategy can reach.

| column | meaning |
| --- | --- |
| `depth` | guesses already played (0 = the root, before the opener) |
| `guesses` | those guesses, in order, space separated |
| `patterns` | the feedback each one got, space separated `byg` strings |
| `n_answers` | answers still possible at this position |
| `guess` | the **optimal next guess** — the tree's move here |
| `solved` | one answer left, so `guess` just plays it |
| `sample` | a few of the possible answers, for identification |

**How it is derived.** Every answer's line records the full path the strategy walks for
that answer, so taking each *prefix* of its (guess, pattern) pairs enumerates the positions
that answer passes through. Answers sharing a prefix are exactly the answers still
consistent with that history — the strategy is deterministic, so two answers part company
only where their feedback differs — which is why counting the lines through a node gives
its candidate count. The last row of a path is the position where one answer remains and
`guess` is that answer.

In [5]:
def read_tree(opener, trees=None):
    """Parse one committed tree; yield (answer, n_guesses, guesses, patterns) per answer.

    `guesses` has n entries and `patterns` has n-1: the final guess is the answer
    itself and carries no pattern (its feedback is all-green by construction)."""
    path = Path(trees or paths.TREES) / f'{opener}.txt.gz'
    if not path.exists():
        raise FileNotFoundError(f'no tree for {opener!r} at {path}')
    with gzip.open(path, 'rt') as f:
        for line in f:
            parts = line.split()
            if not parts:
                continue
            yield parts[0], int(parts[1]), parts[2::2], parts[3::2]


def positions(opener, trees=None, sample=4):
    """Every position reachable under `opener`'s optimal tree, as a DataFrame.

    One row per distinct (guesses so far, patterns so far) history -- see the
    markdown above for the columns and how the counts are derived. `sample` caps
    how many possible answers each row lists."""
    nodes = {}
    for answer, n, guesses, pats in read_tree(opener, trees):
        for k in range(len(guesses)):            # k pairs played, guesses[k] is next
            key = tuple(zip(guesses[:k], pats[:k]))
            node = nodes.get(key)
            if node is None:
                nodes[key] = node = {'guess': guesses[k], 'answers': []}
            elif node['guess'] != guesses[k]:
                # A tree is a deterministic strategy: one move per position. Two
                # different moves after the same history means it is not one.
                raise ValueError(f'{opener}: position {key} has two moves, '
                                 f'{node["guess"]!r} and {guesses[k]!r}')
            node['answers'].append(answer)

    rows = []
    for key, node in nodes.items():
        answers = node['answers']
        rows.append({
            'depth': len(key),
            'guesses': ' '.join(g for g, _ in key),
            'patterns': ' '.join(p for _, p in key),
            'n_answers': len(answers),
            'guess': node['guess'],
            'solved': len(answers) == 1,
            'sample': ' '.join(sorted(answers)[:sample]),
        })
    return (pd.DataFrame(rows)
              .sort_values(['depth', 'n_answers', 'guesses'], ascending=[True, False, True])
              .reset_index(drop=True))


pos = positions(OPENER)
print(f'{OPENER!r}: {len(pos):,} positions, depth {pos["depth"].min()}-{pos["depth"].max()}')
display(pos.head(50))

'salet': 2,391 positions, depth 0-4


,depth,guesses,patterns,n_answers,guess,solved,sample
0,0,,,2315,salet,False,aback abase abate abbey
1,1,salet,bbbbb,221,courd,False,biddy bingo birch bobby
2,1,salet,bbbyb,165,drone,False,beech beefy begin begun
3,1,salet,bbbgb,107,rownd,False,bicep boney boxer breed
4,1,salet,bybbb,102,brond,False,aback abhor acorn acrid
5,1,salet,bbybb,99,courd,False,blimp blind blink block
6,1,salet,bbbby,86,north,False,birth bitty booth booty
7,1,salet,bybyb,83,beard,False,abide abode above adage
8,1,salet,bgbbb,82,corny,False,bacon baggy banjo baron
9,1,salet,byybb,67,corni,False,afoul alarm album alibi


## Positions at one depth

`DEPTH` guesses are already in, and `guess` is the next one. `DEPTH = 0` is the root — one
position, the empty history, whose move is the opener itself.

In [6]:
DEPTH = 1      # <-- guesses already played

at = pos[pos['depth'] == DEPTH]
live = at[~at['solved']]
print(f'depth {DEPTH}: {len(at):,} positions, {len(live):,} of them with more than one '
      f'answer left, covering {at["n_answers"].sum():,} answers')
display(at.head(50).reset_index(drop=True))

depth 1: 148 positions, 114 of them with more than one answer left, covering 2,315 answers


,depth,guesses,patterns,n_answers,guess,solved,sample
0,1,salet,bbbbb,221,courd,False,biddy bingo birch bobby
1,1,salet,bbbyb,165,drone,False,beech beefy begin begun
2,1,salet,bbbgb,107,rownd,False,bicep boney boxer breed
3,1,salet,bybbb,102,brond,False,aback abhor acorn acrid
4,1,salet,bbybb,99,courd,False,blimp blind blink block
5,1,salet,bbbby,86,north,False,birth bitty booth booty
6,1,salet,bybyb,83,beard,False,abide abode above adage
7,1,salet,bgbbb,82,corny,False,bacon baggy banjo baron
8,1,salet,byybb,67,corni,False,afoul alarm album alibi
9,1,salet,gbbbb,56,unrip,False,scion scoff scoop scorn


### ...grouped by the next guess

The same depth-`DEPTH` positions, collapsed onto the optimal next guess: `n_answers` is the
total answers that guess is played against across every position it serves, `n_patterns` is
how many distinct feedback histories lead to it, and `patterns` lists them (in the same
descending-`n_answers` order as the table above, `|`-separated once a history is more than
one pattern long). Sorted by `n_answers` descending.

In [ ]:
by_guess = (at.groupby('guess', sort=False)
              .agg(n_answers=('n_answers', 'sum'),
                   n_patterns=('patterns', 'size'),
                   patterns=('patterns', ' | '.join))
              .sort_values('n_answers', ascending=False)
              .reset_index())

print(f'depth {DEPTH}: {len(by_guess):,} distinct next guesses over {len(at):,} positions')
display(by_guess)

## Look up one position

Filter by the history you have in hand — it is just a prefix of the `guesses`/`patterns`
columns. The second table is every outcome the optimal next guess can produce from there.

In [ ]:
HISTORY  = 'salet'     # <-- guesses so far, space separated ('' for the root)
FEEDBACK = 'bbbyb'     # <-- their feedback, space separated `byg` strings ('' for the root)

hit = pos[(pos['guesses'] == HISTORY) & (pos['patterns'] == FEEDBACK)]
if hit.empty:
    print(f'no position with guesses {HISTORY!r} and patterns {FEEDBACK!r} in '
          f'{OPENER!r}\'s tree -- unreachable under this strategy, or a typo')
else:
    r = hit.iloc[0]
    print(f'{r["n_answers"]:,} answers left; optimal next guess: {r["guess"]!r}')
    display(hit)
    kids = pos[pos['guesses'] == (HISTORY + ' ' + r['guess']).strip()]
    print(f'{len(kids)} outcome(s) of {r["guess"]!r} here:')
    display(kids.reset_index(drop=True))

## Cross-check the counts

`n_answers` is read off the tree's structure (lines sharing a prefix). This re-derives it
from scratch for a random sample of positions — filtering the 2315 answers by replaying
each guess/feedback pair through `wordle.pattern` — so a misparse shows up as a mismatch
rather than as a plausible-looking wrong number. Same spirit as `tools/checktrees.py`.

In [ ]:
N_CHECK = 200      # <-- how many random positions to re-derive

_answers = wordle.load_words('mystery')
_tiles = {'b': 0, 'y': 1, 'g': 2}

def answers_after(guesses, patterns):
    """The answers consistent with a history, from the feedback function alone."""
    alive = list(_answers)
    for g, p in zip(guesses, patterns):
        want = sum(_tiles[c] * 3 ** (wordle.WORD_LEN - 1 - i) for i, c in enumerate(p))
        alive = [a for a in alive if wordle.pattern(g, a) == want]
    return alive

rng = np.random.default_rng(0)                       # reproducible sample
idx = rng.choice(len(pos), size=min(N_CHECK, len(pos)), replace=False)
bad = []
for i in idx:
    r = pos.iloc[int(i)]
    n = len(answers_after(r['guesses'].split(), r['patterns'].split()))
    if n != r['n_answers']:
        bad.append((r['guesses'], r['patterns'], r['n_answers'], n))

print(f'checked {len(idx)} positions of {OPENER!r}: '
      + ('all counts agree' if not bad else f'{len(bad)} MISMATCH(ES)'))
for g, p, tree_n, real_n in bad[:10]:
    print(f'  {g} | {p}: table says {tree_n}, filter says {real_n}')